# OneVoice V2 — Export SenseVoice English fine-tuned sang ONNX

Notebook này chỉ export checkpoint đã benchmark đạt (`model.pt.ep10`) thành bundle ONNX FP32 mới trên Drive. Không train lại, không thay ONNX runtime hiện hành và không ghi đè checkpoint PyTorch.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
CHECKPOINT = WORK_ROOT / 'models/sensevoice_en_construction_v1/model.pt.ep10'
OUTPUT_DIR = WORK_ROOT / 'models/sensevoice_en_construction_v1_onnx_fp32'
STAGE_DIR = Path('/content/onevoice_sensevoice_export_stage')
MODEL_CACHE = WORK_ROOT / 'model_cache/modelscope'

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['MODELSCOPE_CACHE'] = str(MODEL_CACHE)
# Install FunASR first, then pin the final matching torch/torchaudio pair.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy==2.2.6', 'funasr>=1.4.3', 'modelscope', 'soundfile', 'onnx', 'onnxscript'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', 'torch==2.10.0', 'torchaudio==2.10.0', '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
import torch, torchaudio
if not (torch.__version__.startswith('2.10.0') and torchaudio.__version__.startswith('2.10.0')):
    raise RuntimeError(f'PyTorch ABI pin failed: torch={torch.__version__}, torchaudio={torchaudio.__version__}. Restart runtime, then rerun this cell once.')
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f'Missing validated checkpoint: {CHECKPOINT}')
print('Checkpoint:', CHECKPOINT, f'({CHECKPOINT.stat().st_size / 1024**2:.1f} MB)')
print('Output ONNX bundle:', OUTPUT_DIR)


In [ ]:
command = [
    sys.executable, 'scripts/export_sensevoice_checkpoint_onnx.py',
    '--checkpoint', str(CHECKPOINT),
    '--output-dir', str(OUTPUT_DIR),
    '--stage-dir', str(STAGE_DIR),
    '--cache-dir', str(MODEL_CACHE),
    '--device', 'cpu',
]
print('> ' + ' '.join(command), flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError('ONNX export failed; the full traceback is printed above. Do not use a partial output directory.')


In [ ]:
manifest = OUTPUT_DIR / 'export_manifest.json'
if not manifest.is_file():
    raise FileNotFoundError('Export manifest is missing; do not benchmark or deploy this ONNX bundle.')
display(json.loads(manifest.read_text(encoding='utf-8')))
print('Next gate: benchmark this ONNX bundle against the same held-out EN clean/noisy test before replacing any runtime artifact.')
